In [ ]:
import psycopg2
from psycopg2 import sql
from psycopg2.extras import execute_values, Json
import logging
import json
import html
import re

In [ ]:
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

In [ ]:
DB_PARAMS = {
    'dbname':   'thecall',
    'user':     'postgres',
    'password': 'password',
    'host':     'localhost',
    'port':     '5432'
}

TABLE_SCHEMA = 'articles'
TABLE_NAME   = 'filelist'

# Rows per commit. Reduce if you hit memory limits.
BATCH_SIZE = 500

# ── Entity cleaning ───────────────────────────────────────────────────────────
_ENTITY_MAP_PLAIN = [
    (re.compile(r'&(?:amp;)?hTab;'),       '  '),
    (re.compile(r'&(?:amp;)?punctSpace;'), ' '),
    (re.compile(r'&(?:amp;)?nbHyphen;'),   '-'),
    (re.compile(r'&(?:amp;)?nbSpace;'),    ' '),
    (re.compile(r'&(?:amp;)?hairsp;'),     ''),
    (re.compile(r'&(?:amp;)?thinsp;'),     ' '),
    (re.compile(r'&(?:amp;)?numsp;'),      ' '),
    (re.compile(r'&(?:amp;)?shy;'),        ''),
    (re.compile(r'&(?:amp;)?[a-zA-Z][a-zA-Z0-9]{2,10};'), ''),  # catch-all (space bug fixed)
]

_ENTITY_MAP_HTML = [
    (re.compile(r'&(?:amp;)?hTab;'),       '&emsp;'),
    (re.compile(r'&(?:amp;)?punctSpace;'), '&#x2008;'),
    (re.compile(r'&(?:amp;)?nbHyphen;'),   '&#x2011;'),
    (re.compile(r'&(?:amp;)?nbSpace;'),    '&nbsp;'),
    (re.compile(r'&(?:amp;)?hairsp;'),     '&#x200a;'),
    (re.compile(r'&(?:amp;)?thinsp;'),     '&thinsp;'),
    (re.compile(r'&(?:amp;)?numsp;'),      '&#x2007;'),
    (re.compile(r'&(?:amp;)?shy;'),        '&shy;'),
    (re.compile(r'&(?:amp;)?[a-zA-Z][a-zA-Z0-9]{2,10};'), ''),  # catch-all
]


In [ ]:
# ── Database helpers ──────────────────────────────────────────────────────────

def connect_to_db(params):
    try:
        logging.info("Connecting to PostgreSQL...")
        conn = psycopg2.connect(**params)
        conn.autocommit = False
        logging.info("Connected.")
        return conn
    except psycopg2.Error as e:
        logging.error(f"Connection failed: {e}")
        return None




In [ ]:
def fetch_batch(conn, schema, table, batch_size):
    """Fetch rows where json_raw is populated but json_unified not yet written."""
    with conn.cursor() as cur:
        cur.execute(
            sql.SQL("""
                SELECT id, json_raw
                FROM {}.{}
                WHERE json_raw IS NOT NULL
                  AND json_unified IS NULL
                LIMIT %s
            """).format(sql.Identifier(schema), sql.Identifier(table)),
            (batch_size,)
        )


In [ ]:
def update_batch(conn, schema, table, rows):
    if not rows:
        return

    # Keep this as is; Json(dict) handles the serialization
    values = [(Json(r[0]), r[1]) for r in rows]

    with conn.cursor() as cur:
        execute_values(
            cur,
            sql.SQL("""
                UPDATE {}.{} AS t
                SET json_unified = data.v::jsonb
                FROM (VALUES %s) AS data(v, id)
                WHERE t.id = data.id::integer
            """).format(
                sql.Identifier(schema),
                sql.Identifier(table)
            ).as_string(conn),
            values
        )
    conn.commit()
    logging.info(f"  Committed {len(rows)} rows.")

In [ ]:
# ── Format detection ──────────────────────────────────────────────────────────

def detect_format(raw: dict) -> str:
    """
    Returns 'quarkxpress' or 'indesign'.
    QXP json has a 'boxes' key; InDesign has 'pages'.
    """
    if 'boxes' in raw:
        return 'quarkxpress'
    if 'pages' in raw:
        return 'indesign'
    return 'unknown'

In [ ]:
# ── Normalisation: QuarkXPress ────────────────────────────────────────────────

def normalise_quark(raw: dict, db_id: int) -> dict:
    headline_frames, body_frames = [], []

    for box in raw.get('boxes', []):
        paras = [
            {'style': None, 'text': _extract_para_text(p)}
            for p in box.get('paragraphs', [])
            if _extract_para_text(p).strip()
        ]
        if not paras:
            continue

        # Skip single-paragraph slug/label boxes (e.g. "C0307-Little Stars")
        if len(paras) == 1 and _is_slug(paras[0]['text']):
            continue

        frame = {
            'frame_id':   box.get('box_id'),
            'bounds':     None,
            'paragraphs': paras,
            'text':       '\n'.join(p['text'] for p in paras),
        }

        # Classify: no QXP entities + short = headline/subhead
        has_entity = any('&' in p['text'] for p in paras)
        if not has_entity and len(paras) <= 4:
            headline_frames.append(frame)
        else:
            body_frames.append(frame)

    return {
        'db_id':         db_id,
        'serial':        raw.get('serial'),
        'source_file':   raw.get('source_file'),
        'source_format': 'quarkxpress',
        'layout_name':   raw.get('layout_name'),
        'frames':        headline_frames + body_frames,  # headlines first
    }


In [ ]:
def clean_entities(text: str, entity_map: list) -> str:
    """Apply an entity substitution map to a string. Replaces clean_qxp_entities."""
    for pattern, replacement in entity_map:
        text = pattern.sub(replacement, text)
    return text

In [ ]:
def _extract_para_text(p) -> str:
    """Accept a bare string or a {'text': ...} dict."""
    if isinstance(p, dict):
        return str(p.get('text') or p.get('content') or '')
    return str(p)

def _is_slug(text: str) -> bool:
    """True for file-reference identifiers like 'C0307-Little Stars', 'M0822-Admin Assistant'."""
    return bool(re.match(r'^[A-Z][A-Z0-9]{0,5}[-_]\S', text.strip()))



In [ ]:
# ── Normalisation: InDesign ───────────────────────────────────────────────────

def normalise_indesign(raw: dict, db_id: int) -> dict:
    doc_name    = raw.get('document_name', '')
    layout_name = re.sub(r'\.indd$', '', doc_name, flags=re.IGNORECASE)

    frames = []
    for page in raw.get('pages', []):
        for frame in page.get('frames', []):
            if _is_slug_indesign_frame(frame):
                continue
            paras = [
                {'style': _extract_para_text(p) if not isinstance(p, dict) else p.get('style'),
                 'text':  _extract_para_text(p)}
                for p in frame.get('paragraphs', [])
                if _extract_para_text(p).strip()
            ]
            if not paras:
                continue
            frames.append({
                'frame_id':   None,
                'bounds':     frame.get('bounds'),
                'paragraphs': paras,
                'text':       '\n'.join(p['text'] for p in paras),
            })

    frames.sort(key=lambda fr: (
        fr['bounds'][0] if fr['bounds'] else 0,
        fr['bounds'][1] if fr['bounds'] else 0,
    ))

    return {
        'db_id':         db_id,
        'serial':        None,
        'source_file':   raw.get('source_file', doc_name),
        'source_format': 'indesign',
        'layout_name':   layout_name,
        'frames':        frames,
    }

In [ ]:
def _is_slug_indesign_frame(frame: dict) -> bool:
    """Thin single-paragraph frames containing a file/job slug."""
    b = frame.get('bounds') or []
    if len(b) < 4:
        return False
    height = b[2] - b[0]
    paras  = frame.get('paragraphs', [])
    if height < 0.3 and len(paras) == 1:
        return _is_slug(_extract_para_text(paras[0]))
    return False

In [ ]:
# ── Text rendering ────────────────────────────────────────────────────────────

def make_text_plain(unified: dict) -> str:
    parts = []
    if unified.get('layout_name'):
        t = clean_entities(unified['layout_name'], _ENTITY_MAP_PLAIN)
        if t.strip():
            parts.append(f'<p>{t}</p>')
    for frame in unified.get('frames', []):
        for para in frame.get('paragraphs', []):
            t = clean_entities(_extract_para_text(para['text']), _ENTITY_MAP_PLAIN).strip()
            if t:
                parts.append(f'<p>{t}</p>')
    return '\n'.join(parts)


def make_text_html(unified: dict) -> str:
    """
    Self-contained HTML fragment ready for direct insertion into a web page.

    Pipeline per paragraph:
      1. html.escape()        — convert < > & " to safe HTML entities
      2. clean_qxp_entities() — replace QXP proprietary entities with Unicode

    Step 1 must come first so that legitimate & characters introduced by
    html.escape are not then consumed by the entity cleaner.

    Structure:
        <article class="ad-card" data-format="quarkxpress" data-id="123">
          <h2 class="ad-title">Layout name</h2>
          <div class="ad-frame">
            <p class="ad-para" data-style="pr-Normal">paragraph text</p>
          </div>
        </article>
    """
    fmt    = unified.get('source_format', 'unknown')
    db_id  = unified.get('db_id', '')
    serial = unified.get('serial') or ''
    title = clean_entities(html.escape(unified.get('layout_name') or ''), _ENTITY_MAP_HTML)

    parts = []
    parts.append(
        f'<article class="ad-card" '
        f'data-format="{fmt}" '
        f'data-id="{db_id}" '
        f'data-serial="{serial}">'
    )
    if title:
        parts.append(f'  <h2 class="ad-title">{title}</h2>')

    for frame in unified.get('frames', []):
        frame_id = html.escape(str(frame.get('frame_id') or ''))
        parts.append(f'  <div class="ad-frame" data-frame-id="{frame_id}">')
        for para in frame.get('paragraphs', []):
            # Step 1: escape HTML special chars
            text  = html.escape(para['text'].strip())
            # Step 2: clean QXP entities that survived into the text
            text = clean_entities(text, _ENTITY_MAP_HTML)
            style = html.escape(str(para.get('style') or ''))
            if text:
                parts.append(
                    f'    <p class="ad-para" data-style="{style}">{text}</p>'
                )
        parts.append('  </div>')

    parts.append('</article>')
    return '\n'.join(parts)

In [ ]:
# ── Core transform ────────────────────────────────────────────────────────────

def transform(db_id: int, json_raw) -> dict | None:
    """
    Accepts the raw value from the DB.
    psycopg2 auto-parses jsonb columns to dicts; text/json columns arrive
    as strings and are parsed here. Returns the unified dict, or None on error.
    """
    try:
        raw = json_raw if isinstance(json_raw, dict) else json.loads(json_raw)
    except (json.JSONDecodeError, TypeError) as e:
        logging.warning(f"  id={db_id}: cannot parse json_raw — {e}")
        return None

    fmt = detect_format(raw)
    if fmt == 'quarkxpress':
        unified = normalise_quark(raw, db_id)
    elif fmt == 'indesign':
        unified = normalise_indesign(raw, db_id)
    else:
        logging.warning(f"  id={db_id}: unknown format, skipping.")
        return None

    unified['text_plain'] = make_text_plain(unified)
    unified['text_html']  = make_text_html(unified)
    return unified

In [ ]:
# ── Main ──────────────────────────────────────────────────────────────────────

def main():
    conn = connect_to_db(DB_PARAMS)
    if not conn:
        return

    total_processed = 0
    total_errors    = 0

    try:
        while True:
            rows = fetch_batch(conn, TABLE_SCHEMA, TABLE_NAME, BATCH_SIZE)
            if not rows:
                break

            logging.info(f"Processing batch of {len(rows)} rows...")
            updates = []

            for db_id, json_raw in rows:
                unified = transform(db_id, json_raw)
                if unified is None:
                    total_errors += 1
                    continue
                updates.append((unified, db_id))

            update_batch(conn, TABLE_SCHEMA, TABLE_NAME, updates)
            total_processed += len(updates)

    finally:
        conn.close()
        logging.info("Database connection closed.")
        logging.info(f"Done. Processed: {total_processed}  Errors: {total_errors}")


if __name__ == '__main__':
    main()

In [ ]:
# ── ONE-TIME: Reset json_unified so the corrected script can reprocess all rows ──
# Run this cell ONCE before running main().
# It sets json_unified back to NULL for every row so fetch_batch picks them up again.
# Comment it out after the first run.

# conn = connect_to_db(DB_PARAMS)
# if conn:
#     with conn.cursor() as cur:
#         cur.execute(
#             sql.SQL("UPDATE {}.{} SET json_unified = NULL WHERE json_raw IS NOT NULL").format(
#                 sql.Identifier(TABLE_SCHEMA), sql.Identifier(TABLE_NAME)
#             )
#         )
#         logging.info(f"Reset {cur.rowcount:,} rows to json_unified = NULL.")
#     conn.commit()
#     conn.close()

In [ ]:
# ── Spot-check a single row without writing to the DB ────────────────────────

CHECK_ID = 415579  # <-- change this to any id in your table

conn = connect_to_db(DB_PARAMS)
if conn:
    with conn.cursor() as cur:
        cur.execute(
            sql.SQL("SELECT id, json_raw FROM {}.{} WHERE id = %s").format(
                sql.Identifier(TABLE_SCHEMA), sql.Identifier(TABLE_NAME)
            ),
            (CHECK_ID,)   # parameter tuple — fixes the missing-arg bug in v2
        )
        row = cur.fetchone()
    conn.close()

    if row:
        result = transform(row[0], row[1])
        if result:
            print("=== UNIFIED (excluding text fields) ===")
            preview = {k: v for k, v in result.items() if k not in ('text_plain', 'text_html')}
            print(json.dumps(preview, indent=2, ensure_ascii=False))

            print("\n=== TEXT_PLAIN ===")
            print(result['text_plain'])

            print("\n=== TEXT_HTML ===")
            print(result['text_html'])
    else:
        print(f"No row found with id={CHECK_ID}")